# Indium: Elemental Text Inspection & Sanitization

This notebook demonstrates how **indium** protects your application from invisible character attacks, visual spoofing (homoglyphs), and text processing bugs caused by complex Unicode sequences.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarsZDF/indium/blob/main/indium_demo.ipynb)

## Installation

In [ ]:
# Install indium
!pip install elemental-indium

## 1. Invisible Characters: The "Ghost" in Your String

Invisible characters (like Zero Width Space) can cause havoc in databases, search indexing, and username validation. They look empty but have length.

In [ ]:
import indium

# A string that LOOKS like "hello world"
ghost_text = "hello​world"

print(f"Original text: '{ghost_text}'")
print(f"Length: {len(ghost_text)} (Wait, why 11? 'hello world' is 10 chars!)")

# Reveal the invisible character
revealed = indium.reveal(ghost_text, format="name")
print(f"\nRevealed: '{revealed}'")

# Sanitize it
clean = indium.sanitize(ghost_text)
print(f"Sanitized: '{clean}'")
print(f"New Length: {len(clean)}")

## 2. Visual Spoofing: Homoglyph Attacks

Attackers use characters from other scripts (Cyrillic, Greek, etc.) that look identical to Latin letters to spoof domains or usernames.

**Can you spot the difference?**

In [ ]:
legit = "paypal.com"
fake  = "pаypal.com"  # Uses Cyrillic 'а' (U+0430)

print(f"Legit: {legit}")
print(f"Fake:  {fake}")
print(f"Are they equal? {legit == fake}")

# Use indium.skeleton() to normalize confusables
# 'Skeleton' is a Visual Normalization technique that:
# 1. Decomposes fancy characters (NFKC) -> e.g. 𝐇𝐞𝐥𝐥𝐨 -> Hello
# 2. Maps look-alike characters (Confusables) to prototypes -> e.g. Cyrillic 'а' -> Latin 'a'
skeleton_legit = indium.skeleton(legit)
skeleton_fake = indium.skeleton(fake)

print(f"\nSkeleton Legit: {skeleton_legit}")
print(f"Skeleton Fake:  {skeleton_fake}")
print(f"Match detected? {skeleton_legit == skeleton_fake}")

### Mixed Script Detection
Legitimate domains usually stick to one script. "pаypal" mixes Latin and Cyrillic.

In [ ]:
print(f"Is '{legit}' mixed script? {indium.is_mixed_script(legit)}")
print(f"Is '{fake}' mixed script?  {indium.is_mixed_script(fake)}")

# Analyze scripts
print(f"\nScript blocks in fake domain: {indium.get_script_blocks(fake)}")

## 3. Grapheme Clusters: Don't Break the Emoji Family!

Standard string slicing operates on **code points**, not **visual characters** (graphemes). Slicing a multi-codepoint emoji can leave broken artifacts.

In [ ]:
family = "👨‍👩‍👧"  # Man + ZWJ + Woman + ZWJ + Girl

print(f"Visual length: 1 char")
print(f"Python len(): {len(family)} codepoints")

# WRONG: Naive slicing
broken = family[:2]
print(f"\n❌ Naive slice [:2]: '{broken}' (Broken! Just the Man part + glue)")

# CORRECT: Indium safe truncation
safe = indium.safe_truncate(family, 1)
print(f"✅ Safe truncate(1): '{safe}' (Preserves the whole family)")

### Advanced Grapheme Handling
Indium handles Skin Tones, Flags, and Hangul Syllables correctly according to Unicode UAX #29.

In [ ]:
complex_text = "Hello👋🏽🇺🇸"

# Count visual units
g_count = indium.count_graphemes(complex_text)
print(f"Text: {complex_text}")
print(f"Grapheme count: {g_count} (H, e, l, l, o, 👋🏽, 🇺🇸)")

# Iterate over them
print("\nGraphemes:")
for g in indium.iter_graphemes(complex_text):
    print(f" - {g} (len {len(g)}) ")

## 4. Performance Benchmarks

Indium is optimized for speed, using fast-path checks for common ASCII text.

In [ ]:
import time

iterations = 100_000
text_ascii = "helloworld" * 10
text_unicode = "héllo👋world" * 10

print("Benchmarking is_mixed_script()...")

start = time.perf_counter()
for _ in range(iterations):
    indium.is_mixed_script(text_ascii)
elapsed = time.perf_counter() - start
print(f"ASCII (Fast Path): {elapsed/iterations*1e6:.2f} µs/op")

start = time.perf_counter()
for _ in range(iterations):
    indium.is_mixed_script(text_unicode)
elapsed = time.perf_counter() - start
print(f"Unicode (Full Check): {elapsed/iterations*1e6:.2f} µs/op")

## Summary

✅ **Use indium when**:
- Validating usernames (prevent spoofing/invisibles)
- Truncating user-generated content (prevent broken emoji)
- Analyzing text for security risks (phishing domains)
- Counting "visual" characters for UI limits
- **Sanitizing RAG context windows** (strip invisible prompt injections)
- **Pre-tokenization normalization** (mitigate homoglyph-based adversarial attacks)

❌ **Don't use indium for**:
- Cryptographic hashing
- Rendering text (use a rendering engine)

## Learn More
- **GitHub**: [github.com/MarsZDF/indium](https://github.com/MarsZDF/indium)
- **PyPI**: `pip install elemental-indium`